# Provider personality trait tables

Mirrors the data-loading approach used in `02_plotting_paper.ipynb` (final
table cell) and uses the same models as `14_provider_traits_over_time.ipynb`.

For each provider (Anthropic, OpenAI, Google) generate one LaTeX table
showing the `strength_with_stats` metric across that provider's models for
the top 30 traits (ranked by Max diff). The tables are saved under
`output/tex/appendix/provider_traits_over_time/`.


In [5]:
import importlib
import pathlib

import feedback_forensics as ff
import feedback_forensics.app.plotting.paper as paper_plot

importlib.reload(paper_plot)
importlib.reload(ff)

# Same data file used in the final table cell of 02_plotting_paper.ipynb and
# in 14_provider_traits_over_time.ipynb.
data_path = pathlib.Path(
    "/Users/arduin/main/repos/huggingface/ff-model-personality/data/v2/annotations/combined_ap.json"
)

tex_app_save_path = pathlib.Path("./output/tex/appendix/provider_traits_over_time")
tex_app_save_path.mkdir(parents=True, exist_ok=True)

cache = {}
dataset = ff.DatasetHandler(cache=cache)
dataset.add_data_from_path(data_path)


📜  | INFO | AnnotatedPairs format version: 2.0
📜  | INFO | Removing 222 comparisons with empty responses. Fraction affected: 1.65%
📜  | INFO | Created 26454 annotations for 28 model annotators with 28 reference models in 0.15 seconds
📜  | INFO | Loaded data from path: /Users/arduin/main/repos/huggingface/ff-model-personality/data/v2/annotations/combined_ap.json


In [ ]:
# Models per provider (mirrors notebook 14, uncommented entries only).
# The reference model (gpt-4o-2024-11-20) is included with the OpenAI group
# since that is how it is rendered in 14_provider_traits_over_time.ipynb.
PROVIDER_MODELS = {
    "Anthropic": [
        "openrouter/anthropic/claude-3.7-sonnet",
        "openrouter/anthropic/claude-sonnet-4",
        "openrouter/anthropic/claude-sonnet-4.5",
        "openrouter/anthropic/claude-opus-4.7",
    ],
    "OpenAI": [
        #"openrouter/openai/gpt-4o-2024-11-20",  # reference model in nb14
        "openrouter/openai/gpt-5-chat",
        "openrouter/openai/gpt-5.1-chat",
        "openrouter/openai/gpt-5.3-chat",
    ],
    "Google": [
        "openrouter/google/gemini-2.5-pro",
        "openrouter/google/gemini-3-pro-preview",
        "openrouter/google/gemini-3.1-pro-preview",
    ],
}

# Slug used in output filenames per provider.
PROVIDER_SLUG = {
    "Anthropic": "claude",
    "OpenAI": "gpt",
    "Google": "gemini",
}

# Reuse the friendly-name pattern from 02_plotting_paper.ipynb (cell-7)
# so column headers in the LaTeX tables match the rest of the paper.
REPLACE_DICT = {
    "google": "Google",
    "openai": "OpenAI",
    "mistralai": "Mistral",
    "meta-llama": "Meta",
    "x-ai": "xAI",
    "anthropic": "Anthropic",
    "claude": "Claude",
    "gpt": "GPT",
    "mistral": "Mistral",
    "gemini": "Gemini",
    "grok": "Grok",
    "sonnet": "Sonnet",
    "medium": "Medium",
    "Mistral-Medium": "Medium",
    "opus": "Opus",
    "-preview": "",
}


def prettify_visible_name(name: str) -> str:
    """Replicates the column-header rewrite in 02_plotting_paper.ipynb cell-7."""
    name = name.replace("Model: ", "")
    parts = name.split("/")
    if len(parts) >= 2:
        name = parts[0] + " \\textit{" + parts[1] + "}"
    for key, value in REPLACE_DICT.items():
        name = name.replace(key, value)
    return name


In [7]:
# Build a lookup from model_id -> annotator_key, then for each provider build
# the per-model column ordering we want in the table.
annotator_metadata = dataset.get_available_annotators()

model_to_annotator_key = {
    meta["model_id"]: key
    for key, meta in annotator_metadata.items()
    if meta.get("model_id") is not None
}

for provider, models in PROVIDER_MODELS.items():
    found = [m for m in models if m in model_to_annotator_key]
    missing = [m for m in models if m not in model_to_annotator_key]
    print(f"[{provider}] found {len(found)}/{len(models)} models in dataset")
    for m in missing:
        print(f"    missing: {m}")


[Anthropic] found 4/4 models in dataset
[OpenAI] found 4/4 models in dataset
[Google] found 3/3 models in dataset


In [8]:
# Generate one LaTeX table per provider, showing the strength metric across
# the provider's models. Output goes to output/tex/appendix/provider_traits_over_time/.

metric_name = "strength_with_stats"
TOP_N = 30  # show the top-N traits (by Max diff across the provider's models)

saved_paths = []

for provider, models in PROVIDER_MODELS.items():
    # Restrict the dataset's active annotator columns to this provider's models.
    selected = {
        key: meta
        for key, meta in annotator_metadata.items()
        if meta.get("model_id") in set(models)
    }
    if not selected:
        print(f"[{provider}] no annotators found - skipping")
        continue

    # Apply the friendly column name. Operate on a copy so we don't mutate
    # the dataset's metadata permanently between providers.
    original_names = {}
    for key, meta in selected.items():
        original_names[key] = meta["annotator_visible_name"]
        meta["annotator_visible_name"] = prettify_visible_name(meta["annotator_visible_name"])

    try:
        dataset.set_annotator_cols(annotator_keys=list(selected.keys()))
        df = dataset.get_annotator_metrics_df(
            metric_name=metric_name,
            index_col_name="Generate a response that...",
        )

        # Order the model columns the same way they are listed in PROVIDER_MODELS
        # (which mirrors release-date order in nb14). Keep the leading trait
        # column and trailing "Max diff" column in place. The visible names
        # were prettified in-place above, so just read them back from selected.
        ordered_model_cols = [
            selected[model_to_annotator_key[m]]["annotator_visible_name"]
            for m in models
            if m in model_to_annotator_key
        ]
        existing_cols = list(df.columns)
        leading = [c for c in existing_cols if c == "Generate a response that..."]
        trailing = [c for c in existing_cols if c == "Max diff"]
        df = df[leading + ordered_model_cols + trailing]

        # Bonferroni correction is over the full hypothesis space, not the
        # truncated one — keep this based on the full df length.
        num_tested_hypotheses = len(df) * len(ordered_model_cols)

        df_top = df.head(TOP_N)

        latex_str = paper_plot.get_latex_table_from_metrics_df(
            metrics_df=df_top,
            title=f"{provider} model personality traits (Strength)",
            first_col_width=0.23,
            num_tested_hypotheses=num_tested_hypotheses,
        )

        slug = PROVIDER_SLUG[provider]
        out_path = tex_app_save_path / f"{slug}_strength.tex"
        with open(out_path, "w", encoding="utf-8") as f:
            f.write(latex_str)
        saved_paths.append(out_path)
        print(f"[{provider}] wrote {out_path} ({len(df_top)}/{len(df)} traits, {len(ordered_model_cols)} models)")
    finally:
        # Restore the dataset's original visible names so re-running cells stays
        # idempotent and the next provider sees clean metadata.
        for key, original in original_names.items():
            annotator_metadata[key]["annotator_visible_name"] = original

print()
print("Saved tables:")
for p in saved_paths:
    print("  -", p)


📜  | INFO | Setting annotator cols to ['Anthropic \\textit{Claude-Sonnet-4.5}', 'Anthropic \\textit{Claude-Opus-4.7}', 'Anthropic \\textit{Claude-Sonnet-4}', 'Anthropic \\textit{Claude-3.7-Sonnet}']
📜  | WARNING | Reference annotator column '359d7a6d' contains values other than 'text_a' or 'text_b' (Values: Not applicable, text_a). Metrics will be computed on the subset of votes where the reference annotator is 'text_a' or 'text_b'.
📜  | WARNING | Reference annotator column 'c869561c' contains values other than 'text_a' or 'text_b' (Values: Not applicable, text_a). Metrics will be computed on the subset of votes where the reference annotator is 'text_a' or 'text_b'.
📜  | WARNING | Reference annotator column '204e8c2d' contains values other than 'text_a' or 'text_b' (Values: Not applicable, text_a). Metrics will be computed on the subset of votes where the reference annotator is 'text_a' or 'text_b'.
📜  | WARNING | Reference annotator column '81ea1afa' contains values other than 'text_a

In [9]:
# Bonus: total trait drift per provider — sum of per-trait Max diff across
# ALL traits in the dataset (not just the top 30 shown in the LaTeX tables).
# "Max diff" for a given provider/trait is the spread between that
# provider's strongest and weakest model on that trait, so summing it gives
# a one-number summary of "how much has this provider changed across the
# whole trait space".
import pandas as pd

per_provider_summary = {}

for provider, models in PROVIDER_MODELS.items():
    selected_keys = [
        key for key, meta in annotator_metadata.items()
        if meta.get("model_id") in set(models)
    ]
    if not selected_keys:
        print(f"[{provider}] no annotators found - skipping")
        continue

    dataset.set_annotator_cols(annotator_keys=selected_keys)
    df = dataset.get_annotator_metrics_df(
        metric_name="strength_with_stats",
        index_col_name="Generate a response that...",
    )

    max_diff_total = float(df["Max diff"].sum())
    per_provider_summary[provider] = {
        "n_models": len(selected_keys),
        "n_traits": len(df),
        "sum_max_diff": max_diff_total,
        "mean_max_diff": max_diff_total / len(df) if len(df) else 0.0,
    }

summary_df = (
    pd.DataFrame.from_dict(per_provider_summary, orient="index")
    .sort_values("sum_max_diff", ascending=False)
)
print("Total trait drift per provider (sum of per-trait Max diff across all traits):")
print(summary_df.round(3).to_string())
summary_df


📜  | INFO | Setting annotator cols to ['Model: anthropic/claude-sonnet-4.5', 'Model: anthropic/claude-opus-4.7', 'Model: anthropic/claude-sonnet-4', 'Model: anthropic/claude-3.7-sonnet']
📜  | WARNING | Reference annotator column '359d7a6d' contains values other than 'text_a' or 'text_b' (Values: Not applicable, text_a). Metrics will be computed on the subset of votes where the reference annotator is 'text_a' or 'text_b'.
📜  | WARNING | Reference annotator column 'c869561c' contains values other than 'text_a' or 'text_b' (Values: Not applicable, text_a). Metrics will be computed on the subset of votes where the reference annotator is 'text_a' or 'text_b'.
📜  | WARNING | Reference annotator column '204e8c2d' contains values other than 'text_a' or 'text_b' (Values: Not applicable, text_a). Metrics will be computed on the subset of votes where the reference annotator is 'text_a' or 'text_b'.
📜  | WARNING | Reference annotator column '81ea1afa' contains values other than 'text_a' or 'text_b

,n_models,n_traits,sum_max_diff,mean_max_diff
OpenAI,4,40,11.329328,0.283233
Google,3,40,5.318240,0.132956
Anthropic,4,40,5.309456,0.132736
